# 04 — Per-Country Raster Export (Phase 4)

Exports continuous raster surfaces (GeoTIFF) per country, with one band per hazard + exposure layer. Reads `COUNTRY_CONFIGS` for the list of countries to export.

## Inputs

- GEE assets under `projects/unicef-ccri/assets/hazards/` (see `config/hazard_info.json`)
- GEE WorldPop population per country (R2025A v1, 100m, total + male/female + U18 splits)
- GEE admin0/admin1/admin2 boundary FeatureCollections
- `config/hazard_info.json` — hazard asset config

## Outputs

GeoTIFFs to Google Drive (Earth Engine export task), one per country.

## Execution order

Independent — can run any time after assets exist. Typically after notebook 02 for canonical hazard set.


In [1]:
# ============================================================
# CCRI Hazard Statistics Processing in GEE.
# Phase 4: hazard exposure raster export. 
# Author: Angelly Pugliese, Ph.D.
# Date: June 2026
# ============================================================

In [2]:
# ============================================================
# Imports
# ============================================================
import sys
from pathlib import Path

# This notebook lives in handoff/notebooks/. Anchor everything to the
# project root (handoff/) so config/bounds/lib resolve regardless of CWD.
PROJECT_ROOT = Path.cwd().parent          # notebooks/ -> handoff/
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
BOUNDS_DIR = PROJECT_ROOT / "bounds"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import pandas as pd
import ee
from GEE_functions import GEEUtils 
from GEE_functions import Hazard
import datetime as dt

pd.set_option('display.max_columns', 500)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
# ============================================================
# Authenticate and initialize the Earth Engine library
# Define GEE asset path for UNICEF CCRI data
# ============================================================
ee.Authenticate()
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)

In [4]:
prod_date = dt.date.today().isoformat()

In [5]:
PROCESS_SPECIFIC_COUNTRIES = ['SLE'] #ISO3 (not ucode)

# Country-specific configurations
# Sets specific hazards per country, otherwise exports all hazards. 
# Also sets whether to export population exposure for the country.
COUNTRY_CONFIGS = {
    'SLE_V1': {'hazards': ['coastal_flood_100yr'], 'export_exposure': False}, 
}

PROCESS_SPECIFIC_HAZARDS = ['coastal_flood_100yr']
countries_to_process = utils.get_countries_to_process(PROCESS_SPECIFIC_COUNTRIES)
print(f"Countries to process ({len(countries_to_process)}): {countries_to_process}")

Countries to process (1): ['SLE_V1']


In [6]:
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

# Now hazard_list contains Hazard objects
for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}, {hazard.threshold}, {hazard.threshold_operation}")

0. Riverine flood hazard, river_flood_100yr, projects/unicef-ccri/assets/hazards/river_flood_r100, 0.01, gt
1. Coastal flood hazard, coastal_flood_100yr, projects/unicef-ccri/assets/hazards/coastal_flood_r100, 0.0, gt
2. Wind speed, tropical_storm_100yr, projects/unicef-ccri/assets/hazards/storm_giri_rp100, 17.5, gt
3. Agricultural Stress Index, drought_asi, projects/unicef-ccri/assets/hazards/ASI_return_level_100yr, 30.0, gt
4. Heatwave frequency, heatwave_frequency, projects/unicef-ccri/assets/hazards/heatwave_frequency_return_level_100yr, 16.02581287445288, gt
5. Heatwave duration, heatwave_duration, projects/unicef-ccri/assets/hazards/heatwave_duration_return_level_100yr, 94.01266417971051, gt
6. Heatwave severity, heatwave_severity, projects/unicef-ccri/assets/hazards/heatwave_severity_return_level_100yr, 3.659148767122408, gt
7. Extreme hot days, extreme_heat_days, projects/unicef-ccri/assets/hazards/high_temp_degree_days_return_level_100yr, 35.0, gt
8. Fire Intensity, fire_inten

In [7]:
# Helper function to check if hazard should be processed for country
def should_process_hazard(country_ucode, hazard_code, hazard_list_all):
    if country_ucode not in COUNTRY_CONFIGS:
        if hazard_code in PROCESS_SPECIFIC_HAZARDS:
            return True
    return False
    

# Helper function to check if exposure should be exported for country
def should_export_exposure(country_ucode):
    if country_ucode not in COUNTRY_CONFIGS:
        return True  # Export exposure by default
    return COUNTRY_CONFIGS[country_ucode]['export_exposure']


In [9]:
# -------------------------------
# USER CONFIG
# -------------------------------
POP_CLASSES = ["total", "under_18_total"]
EXPORT_FOLDER = "UNICEF_Exposure_Maps"

# -------------------------------
# SharePoint auto-sync helpers
# -------------------------------
import sharepoint_sync as sps

# Drive OAuth desktop-app client + cached token.
DRIVE_CLIENT_SECRET = CONFIG_DIR.parent / "credentials" / "drive_oauth_client.json"
DRIVE_TOKEN_FILE    = CONFIG_DIR.parent / "credentials" / "drive_token.json"

# Authorize Drive up front so Stage 0 can pre-create the per-ucode folders
drive = sps.get_drive_service(DRIVE_CLIENT_SECRET, DRIVE_TOKEN_FILE)

# ucode -> Drive folder id (so we can scope the later download to this run).
drive_folder_ids = {}

# Collected (task, description, country_ucode) for every started export.
submitted_tasks = []

In [ ]:
# -------------------------------
# COUNTRY PROCESSING SINGLE BANDS
# -------------------------------
for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS is not None and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    now = pd.Timestamp.now()
    for country_ucode in countries_to_process:
        # Check if this hazard should be processed for this country
        if not should_process_hazard(country_ucode, hazard.code, hazard_list):
            #print(f"  Skipping {hazard.code} for {country_ucode}")
            continue

        try:

            print(f"Processing for hazard: {hazard.name}")

            # Stage 0: ensure the root-level Drive folder for this ucode exists
            # exactly once, then EE writes into it by name (folder=country_ucode).
            if country_ucode not in drive_folder_ids:
                drive_folder_ids[country_ucode] = sps.ensure_drive_folder(drive, country_ucode)

            hazard_images = utils.define_hazard(hazard)
            country_adm2_fc = utils.get_admin2_boundaries(country_ucode)
            country_fc = utils.get_country_boundaries(country_ucode)

            # Set population assets
            utils.set_population_dicts(country_ucode, country_adm2_fc)

            # Population classes
            pop_info = utils.get_population_classes(
                include_classes=POP_CLASSES
            )

            hazard_raw_clipped = hazard_images["hazard_raw_image"].clip(country_fc)

            # Hazard (raw)
            task = utils.export_singleband_to_drive(
                image=hazard_raw_clipped,
                band_name="hazard_raw",
                region_fc=country_fc,
                country_ucode=country_ucode,
                hazard_code=hazard.code,
                folder=country_ucode,
                return_task=True
            )
            submitted_tasks.append((task, f"{country_ucode}_{hazard.code}_hazard_raw", country_ucode))

            # Exposure layers (only if configured for this country)
            if should_export_exposure(country_ucode):
                for cls, img in pop_info["valid_population_classes"].items():
                    exp_img = img.multiply(hazard_images["hazard_masked_image"])
                    task = utils.export_singleband_to_drive(
                        image=exp_img,
                        band_name=f"exp_{cls}",
                        region_fc=country_fc,
                        country_ucode=country_ucode,
                        hazard_code=hazard.code,
                        folder=country_ucode,
                        return_task=True
                    )
                    submitted_tasks.append((task, f"{country_ucode}_{hazard.code}_exp_{cls}", country_ucode))
        except Exception as e:
            print(f"Error exporting {hazard.name} for country {country_ucode}, error> {e}")

print(f"\nSubmitted {len(submitted_tasks)} export task(s) across {len(drive_folder_ids)} country folder(s).")

Processing for hazard: Coastal flood hazard
📁 Created Drive folder 'SLE_V1' (1D8HqBKCGwjLH-lhozr0WXdaLU192Dtwp).
projects/unicef-ccri/assets/hazards/coastal_flood_r100 is mosaic
Applying no data mask with value: -3.4e+38 and theoretical minimum: 0.0
apply threshold with:  0.0
Export started: SLE_V1_coastal_flood_100yr_hazard_raw

Submitted 1 export task(s) across 1 country folder(s).


## Automated Drive → SharePoint hand-off
**Disclaimer**: Additional Microsoft set up is needed here to enable the automation.

After the EE exports above are submitted, the cells below run the hand-off with no manual
download/upload:

- **Stage B** — wait for every export task to finish.
- **Stage C1** — download the finished GeoTIFFs from the per-`ucode` Drive folder.
- **Stage C2** — push them to SharePoint at
  `Climate & Environment Data Unit/1.Processes/Country engagement/2025/<Country Name>/rasters/`,
  creating folders as needed (`<Country Name>` from `config/countries_info.csv`).
- **Stage D** — delete the local temp copies (Drive is kept as a backup).

**Auth:** the first cell triggers a Microsoft **device-code login** — open the printed URL in a
browser, sign in to the UNICEF tenant online, and enter the code. Re-run each session.

Fill in `TENANT_ID` and `GRAPH_CLIENT_ID` before running (see `lib/sharepoint_sync.py` header).

In [ ]:
# ============================================================
# SharePoint hand-off — CONFIG + auth
# ============================================================

# --- SharePoint target (from UNICEF DAPM site) ---
SP_HOSTNAME    = "unicef.sharepoint.com"
SP_SITE_PATH   = "teams/DAPM"
SP_LIBRARY     = "DocumentLibrary4"
SP_BASE_PATH   = "D&A Library/Climate & Environment Data Unit/1.Processes/Country engagement/2025"
SP_RASTERS_SUBFOLDER = "rasters"

# --- Microsoft Graph auth (device-code, delegated) ---
# TODO: fill these in. TENANT_ID = UNICEF tenant; GRAPH_CLIENT_ID = a public-client
# app registration in that tenant with delegated Sites.ReadWrite.All and device-code
# flow enabled (request from UNICEF IT — a consultant can't self-register one).
TENANT_ID       = "<UNICEF_TENANT_ID>"
GRAPH_CLIENT_ID = "<GRAPH_CLIENT_ID>"


# --- Local staging + cleanup ---
LOCAL_TEMP_DIR = PROJECT_ROOT / "tmp_sharepoint_sync"   # gitignored
DELETE_LOCAL_AFTER_UPLOAD = True
POLL_SECONDS = 30

# ucode -> human-readable country name (for the SharePoint folder name).
ucode_to_name = sps.load_ucode_to_name(CONFIG_DIR / "countries_info.csv")

# Device-code login (prints a URL + code to enter in a browser, online).
graph_token = sps.graph_device_code_token(GRAPH_CLIENT_ID, TENANT_ID)
site_id, drive_id = sps.resolve_site_drive(graph_token, SP_HOSTNAME, SP_SITE_PATH, SP_LIBRARY)
print(f"Resolved SharePoint drive '{SP_LIBRARY}' (drive_id={drive_id}).")

In [ ]:
# ============================================================
# Stage B — wait for the EE export tasks to finish
# ============================================================
tasks_only = [t for (t, _desc, _uc) in submitted_tasks]
completed_descs, failures = sps.wait_for_tasks(tasks_only, poll_seconds=POLL_SECONDS)

# Map each completed description back to its country ucode for the download step.
completed_set = set(completed_descs)
completed_by_ucode = {}
for _task, desc, uc in submitted_tasks:
    if desc in completed_set:
        completed_by_ucode.setdefault(uc, []).append(desc)

print(f"\n{len(completed_descs)} file(s) ready to transfer across "
      f"{len(completed_by_ucode)} country folder(s).")

In [ ]:
# ============================================================
# Stage C + D — download from Drive, upload to SharePoint, clean up local temp
# ============================================================
for ucode, descs in completed_by_ucode.items():
    country_name = ucode_to_name.get(ucode)
    if not country_name:
        print(f"No country name for ucode '{ucode}' in countries_info.csv; skipping.")
        continue

    # C1 — download this country's finished TIFFs from its Drive folder.
    local_files = sps.download_drive_files(
        drive,
        filenames=descs,
        dest_dir=LOCAL_TEMP_DIR / ucode,
        parent_folder_id=drive_folder_ids.get(ucode),
    )
    if not local_files:
        continue

    # C2 + D — upload to .../2025/<Country Name>/rasters/, then delete local temp.
    sps.sync_to_sharepoint(
        token=graph_token,
        drive_id=drive_id,
        sp_base_path=SP_BASE_PATH,
        country_name=country_name,
        local_files=local_files,
        rasters_subfolder=SP_RASTERS_SUBFOLDER,
        delete_local_after=DELETE_LOCAL_AFTER_UPLOAD,
    )

print("\n Drive → SharePoint hand-off complete.")